In [18]:
import yfinance as yf
import pandas as pd
import datetime as dt
import fredapi
import numpy as np
from pandas.core.interchange.dataframe_protocol import DataFrame
from scipy.stats import norm
from riskfree_and_spot import *

from yfinance import tickers

#------------------------
valuation_date = "2026-06-03" #YYYY-MM-DD
ticker = "MSFT"
expiration = "2026-07-30"
strike = 480
option_type = "call" # call / put


#-------------------------

if valuation_date == "today":
    valuation_date = dt.datetime.today().strftime('%Y-%m-%d')

#-------------------------

In [15]:
N = norm.cdf

def bs_call (S, K, T, r, vol):
    d1 = (np.log(S/K) + (r + 0.5 * vol ** 2) * T) / (vol*np.sqrt(T))
    d2 = d1 - vol * np.sqrt(T)
    return S * norm.cdf(d1) - np.exp(-r * T) * K * norm.cdf(d2)

def bs_vega(S, K, T, r, sigma):
    d1 = (np.log(S/K) + (r + 0.5 * sigma ** 2) * T) / (sigma * np.sqrt(T))
    return S * norm.pdf(d1) * np.sqrt(T)

def find_vol(target_value, S, K, T, r, *args):
    MAX_ITERATIONS = 200
    PRECISION = 1.0e-5
    sigma = 0.5
    for i in range (0, MAX_ITERATIONS):
        price = bs_call(S, K, T, r, sigma)
        vega = bs_vega(S, K, T, r, sigma)
        diff = target_value - price
        if (abs(diff) < PRECISION):
            return sigma
        sigma = sigma + diff/vega
    return sigma

In [21]:
S = get_spot_price_data(ticker, valuation_date)
K = strike
T = option_tenor_calc(valuation_date, expiration)
r = get_risk_free_rate(valuation_date, expiration)
vol = 0.25

V_market = bs_call(S, K, T, r, vol)
implied_vol = find_vol(V_market, S, K, T, r)
print("implied vol: %.2f%%" % (implied_vol*100))
print("market price = %.2f" % V_market)
print("model price = %.2f" % bs_call(S, K, T, r, implied_vol))
print(f"S:{S}")
print(f"K:{K}")
print(f"T:{T}")
print(f"r:{r}")
print(f"vol:{vol}")
print(f"v_market{V_market}")
print(f"implied_vol{implied_vol}")

[*********************100%***********************]  1 of 1 completed


implied vol: 25.00%
market price = 2.85
model price = 2.85
S:369.3699951171875
K:480
T:0.5013698630136987
r:0.037198904109589044
vol:0.25
v_market2.8547129190912983
implied_vol0.2500000453084516
